# Tutorial: Using MediaPipe GStreamer Plugins with Python

This notebook demonstrates how to use the `facelandmarks`, `mozza_mp`, and `mozza_mp_gpu` plugins using the provided Python wrapper script `mozza_process.py`.

## 1. Prerequisites

To run this tutorial, you need:
- **Docker** installed and running.
- **Python 3** installed locally.
- The unified Docker image built: `docker build -t mp_plugins:latest .`
- (Optional) **NVIDIA GPU** with `nvidia-container-toolkit` for GPU acceleration.

> **Tip:** If you don't require real-time performance or want to avoid the complexity of GPU configuration, we highly recommend using the **CPU mode** (`--mode cpu`). It is much easier to set up and perfectly suitable for processing pre-recorded video files where latency is not a critical factor.

In [ ]:
import os
import sys

# Setup paths relative to the notebook location
tutorial_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(tutorial_dir, ".."))
wrapper_script = os.path.join(root_dir, "mozza_process.py")
assets_dir = os.path.join(root_dir, "assets")
model_path = os.path.join(root_dir, "face_landmarker.task")
deform_path = os.path.join(root_dir, "smile.dfm")

print(f"Root directory: {root_dir}")

## 2. Setup Models and Assets

First, let's ensure we have the necessary MediaPipe model. Run the following script to download `face_landmarker.task` if it's not already present in the root directory.

In [ ]:
!cd {root_dir} && chmod +x download_face_landmarker_model.sh && ./download_face_landmarker_model.sh

## 3. Basic Face Landmark Detection

The simplest mode is `landmarks`, which just overlays the detected landmarks on the input file. We will use the CPU by default here.

In [ ]:
input_img = os.path.join(assets_dir, "test_image.jpg")
output_img = os.path.join(tutorial_dir, "tutorial_landmarks.png")

!python3 {wrapper_script} --input {input_img} --output {output_img} --mode landmarks --model-path {model_path}

## 4. Facial Deformation (The "Smile" Transformation)

Now, let's apply a deformation using the `smile.dfm` rule file. We will compare CPU and GPU modes.

### CPU Mode (Recommended for simple setup)
This uses the `mozza_mp` plugin. We'll increase the `alpha` to 2.0 for a more pronounced effect and use `per-group-roi` for better localized warping.

In [ ]:
output_cpu = os.path.join(tutorial_dir, "tutorial_smile_cpu.png")

!python3 {wrapper_script} --input {input_img} --output {output_cpu} \
    --mode cpu --deform {deform_path} --alpha 2.0 --warp-mode per-group-roi --show-landmarks false --model-path {model_path}

### GPU Mode (Maximum Performance)
This uses the `mozza_mp_gpu` plugin. Note that for this to work, you must have an NVIDIA GPU and the corresponding ONNX models in the same folder as the `.task` file.

In [ ]:
output_gpu = os.path.join(tutorial_dir, "tutorial_smile_gpu.png")

# Note: This might fail if you don't have a GPU configured.
!python3 {wrapper_script} --input {input_img} --output {output_gpu} \
    --mode gpu --deform {deform_path} --alpha 2.0 --warp-mode per-group-roi --show-landmarks false --model-path {model_path}

## 5. Processing Video

The wrapper works exactly the same way for video files. Let's process a short video example using the CPU plugin.

In [ ]:
input_video = os.path.join(assets_dir, "dynamic_video.mp4")
output_video = os.path.join(tutorial_dir, "tutorial_video_smile.mp4")

!python3 {wrapper_script} --input {input_video} --output {output_video} \
    --mode cpu --deform {deform_path} --alpha 1.5 --warp-mode per-group-roi --show-landmarks false --model-path {model_path}

## 6. Summary of Key Parameters

- `--input`: Path to your image or video.
- `--mode`: `cpu` (MediaPipe), `gpu` (TensorRT), or `landmarks` (overlay only).
- `--deform`: Path to the `.dfm` file containing warping rules.
- `--alpha`: Intensity of the effect (e.g., 2.0 for double intensity).
- `--warp-mode`: Use `per-group-roi` to ensure transformations are localized to specific facial features.

For a full list of options, run `python3 mozza_process.py --help`.